<a href="https://colab.research.google.com/github/Jaguar838/ml-zoomcamp/blob/master/HW/hw10/hw_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Homework

In this homework, we'll deploy the lead scoring model from the homework 5.

We already have a docker image for this model - we'll use it for 
deploying the model to Kubernetes.

In [ ]:
#!git clone https://github.com/DataTalksClub/machine-learning-zoomcamp.git

Перейдіть у папку `course-zoomcamp/cohorts/2025/05-deployment/homework`

In [2]:
!ls -a

.		 Dockerfile_full  main.py	   q3_test.py	  q6_test.py
..		 Dockerfile_hw	  pipeline_v1.bin  q4_predict.py  uv.lock
.python-version  README.md	  pipeline_v2.bin  q4_test.py
Dockerfile_base  hw_10.ipynb	  pyproject.toml   q6_predict.py


In [ ]:
!docker build -f Dockerfile_full -t zoomcamp-model:3.13.10-hw10 .

In [4]:
!docker images zoomcamp-model:3.13.10-hw10

                                                            i Info →   U  In Use
IMAGE                         ID             DISK USAGE   CONTENT SIZE   EXTRA
zoomcamp-model:3.13.10-hw10   54e384181eaf        565MB          135MB        


## Question 1

Запустіть його для локального тестування:

In [ ]:
!docker run -it --rm -p 9696:9696 zoomcamp-model:3.13.10-hw10



В іншому терміналі виконайте файл `q6_test.py`:

```bash
python q6_test.py
```

Ви побачите таке:

```python
{'conversion_probability': <value>, 'conversion': False}
```

Тут `<value>` – це ймовірність підписки. Оберіть правильне значення:

* 0.29
* 0.49
* 0.69
* 0.89



In [6]:
!python q6_test.py

{'conversion_probability': 0.49999999999842815, 'conversion': False}


Answer 1: 0.49

## Installing `kubectl` and `kind`

You need to install:

* `kubectl` - https://kubernetes.io/docs/tasks/tools/ (you might already have it - check before installing)
* `kind` - https://kind.sigs.k8s.io/docs/user/quick-start/

(used install using apt for kubectl, and install from repo for kind)

## Question 2

What's the version of `kind` that you have?

Use `kind --version` to find out.


In [7]:
!kind --version

kind version 0.16.0


Answer 2:  0.16.0

## Creating a cluster

Now let's create a cluster with `kind`:

In [8]:
!kind create cluster

Creating cluster "kind" ...
 ✓ Ensuring node image (kindest/node:v1.25.2) 🖼7l
 ✓ Preparing nodes 📦 7l
 ✓ Writing configuration 📜7l
 ✓ Starting control-plane 🕹️7l
 ✓ Installing CNI 🔌7l
 ✓ Installing StorageClass 💾7l
Set kubectl context to "kind-kind"
You can now use your cluster with:

kubectl cluster-info --context kind-kind

Not sure what to do next? 😅  Check out https://kind.sigs.k8s.io/docs/user/quick-start/


Creating cluster "kind" ...
 ✓ Ensuring node image (kindest/node:v1.25.2) 🖼
 ✓ Preparing nodes 📦
 ✓ Writing configuration 📜
 ✓ Starting control-plane 🕹️
 ✓ Installing CNI 🔌
 ✓ Installing StorageClass 💾
Set kubectl context to "kind-kind"
You can now use your cluster with:

kubectl cluster-info --context kind-kind


In [ ]:
!kubectl cluster-info

Kubernetes control plane is running at https://127.0.0.1:36849
CoreDNS is running at https://127.0.0.1:36849/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy

To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.


## Question 3

What's the smallest deployable computing unit that we can create and manage
in Kubernetes (`kind` in our case)?

* Node
* Pod
* Deployment
* Service

**Answer: Pod**

# Question 4

Now let's test if everything works. Use `kubectl` to get the list of running services.

What's the `Type` of the service that is already running there?

* ClusterIP
* NodePort
* LoadBalancer
* ExternalName


In [9]:
!kubectl get services

NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE
kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   60s


**Answer is: The service already running is ClusterIP**

## Question 5

To be able to use the docker image we previously created (`svizor/zoomcamp-model:3.11.5-hw10`),
we need to register it with `kind`.

What's the command we need to run for that?

* `kind create cluster`
* `kind build node-image`
* `kind load docker-image`
* `kubectl apply`


In [10]:
!kind load docker-image zoomcamp-model:3.13.10-hw10


Image: "zoomcamp-model:3.13.10-hw10" with ID "sha256:54e384181eafae3d6d9b599741b10c2873b131567da7a60ddfff60de97ec83dd" not yet present on node "kind-control-plane", loading...


**Answer: kind load docker-image**

## Question 6

Now let's create a deployment config (e.g. `deployment.yaml`):

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: subscription
spec:
  selector:
    matchLabels:
      app: subscription
  replicas: 1
  template:
    metadata:
      labels:
        app: subscription
    spec:
      containers:
      - name: subscription
        image: <Image>
        resources:
          requests:
            memory: "64Mi"
            cpu: "100m"            
          limits:
            memory: <Memory>
            cpu: <CPU>
        ports:
        - containerPort: <Port>
```

Replace `<Image>`, `<Memory>`, `<CPU>`, `<Port>` with the correct values.

What is the value for `<Port>`?


Apply this deployment using the appropriate command and get a list of running Pods.
You can see one running Pod.

```
        image: zoomcamp-model:3.13.10-hw10
        resources:
          requests:
            memory: "64Mi"
            cpu: "100m"            
          limits:
            memory: "128Mi"
            cpu: "250m"
        ports:
        - containerPort: 9696
```

**Answer: The value for port is 9696

In [11]:
!kubectl get pods --all-namespaces

NAMESPACE            NAME                                         READY   STATUS    RESTARTS   AGE
kube-system          coredns-565d847f94-njcl7                     1/1     Running   0          6m37s
kube-system          coredns-565d847f94-wmgf6                     1/1     Running   0          6m37s
kube-system          etcd-kind-control-plane                      1/1     Running   0          6m50s
kube-system          kindnet-7pbp8                                1/1     Running   0          6m37s
kube-system          kube-apiserver-kind-control-plane            1/1     Running   0          6m50s
kube-system          kube-controller-manager-kind-control-plane   1/1     Running   0          6m50s
kube-system          kube-proxy-zxlsp                             1/1     Running   0          6m37s
kube-system          kube-scheduler-kind-control-plane            1/1     Running   0          6m50s
local-path-storage   local-path-provisioner-684f458cdd-tmqvf      1/1     Running   0        

In [12]:
# !kubectl get pods --all-namespaces
!kubectl apply -f deployment.yaml

deployment.apps/subscription created


In [1]:
!kubectl get deployment

NAME           READY   UP-TO-DATE   AVAILABLE   AGE
subscription   1/1     1            1           77m


Yes there is one running pod as expected

In [2]:
!kubectl get pod

NAME                            READY   STATUS    RESTARTS   AGE
subscription-6d5867d9cb-pn98h   1/1     Running   0          77m


## Question 7

Let's create a service for this deployment (`service.yaml`):

```yaml
apiVersion: v1
kind: Service
metadata:
  name: <Service name>
spec:
  type: LoadBalancer
  selector:
    app: <???>
  ports:
  - port: 80
    targetPort: <PORT>
```

Fill it in. What do we need to write instead of `<???>`?

**Answer: subscription**

Apply this config file.

my service.yaml:

```
apiVersion: v1
kind: Service
metadata:
  name: subscription-service
spec:
  type: LoadBalancer
  selector:
    app: subscription
  ports:
  - port: 80
    targetPort: 9696
```

In [15]:
!kubectl apply -f service.yaml

service/subscription-service created


In [3]:
!kubectl get service

NAME                   TYPE           CLUSTER-IP     EXTERNAL-IP   PORT(S)        AGE
kubernetes             ClusterIP      10.96.0.1      <none>        443/TCP        85m
subscription-service   LoadBalancer   10.96.219.16   <pending>     80:32343/TCP   68m
   TYPE           CLUSTER-IP     EXTERNAL-IP   PORT(S)        AGE
kubernetes             ClusterIP      10.96.0.1      <none>        443/TCP        85m
subscription-service   LoadBalancer   10.96.219.16   <pending>     80:32343/TCP   68m


## Testing the service

We can test our service locally by forwarding the port 9696 on our computer
to the port 80 on the service:

In [ ]:
!kubectl port-forward service/subscription-service 9696:80


Run `q6_test.py` (from the homework 5) once again to verify that everything is working.
You should get the same result as in Question 1.

In [ ]:
!python q6_test.py

# Autoscaling

Now we're going to use a[HorizontalPodAutoscaler](https: // kubernetes.io/docs/tasks/run-application/horizontal-pod-autoscale-walkthrough/)
(HPA for short) that automatically updates a workload resource(such as our deployment),
with the aim of automatically scaling the workload to match demand.

Use the following command to create the HPA:

```bash
kubectl autoscale deployment subscription --name subscription-hpa --cpu-percent=20 --min=1 --max=3
```

You can check the current status of the new HPA by running:

```bash
kubectl get hpa
```

The output should be similar to the next:

```bash
NAME               REFERENCE                TARGETS          MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   <unknown>/20%   1         3         1          26s
```

`TARGET` column shows the average CPU consumption across all the Pods controlled by the corresponding deployment.
Current CPU consumption is about 0 % as there are no clients sending requests to the server.
>
>Note: In case the HPA instance doesn't run properly, try to install the latest Metrics Server release
> from the `components.yaml` manifest:
> ```bash
> kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml
>```
serviceaccount/metrics-server created
clusterrole.rbac.authorization.k8s.io/system:aggregated-metrics-reader created
clusterrole.rbac.authorization.k8s.io/system:metrics-server created
rolebinding.rbac.authorization.k8s.io/metrics-server-auth-reader created
clusterrolebinding.rbac.authorization.k8s.io/metrics-server:system:auth-delegator created
clusterrolebinding.rbac.authorization.k8s.io/system:metrics-server created
service/metrics-server created
deployment.apps/metrics-server created
apiservice.apiregistration.k8s.io/v1beta1.metrics.k8s.io created

In [ ]:
!kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml

In [10]:
!kubectl delete hpa subscription-hpa

horizontalpodautoscaler.autoscaling "subscription-hpa" deleted


In [12]:
!kubectl autoscale deployment subscription --name subscription-hpa --cpu-percent=20 --min=1 --max=3

horizontalpodautoscaler.autoscaling/subscription-hpa autoscaled


In [20]:
!kubectl get hpa

NAME               REFERENCE                 TARGETS   MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   3%/20%    1         3         1          58m


In [16]:
!kubectl get pods -n kube-system | grep metrics-server

metrics-server-5c474f94c4-nrwmp              0/1     Running   0          10m


In [ ]:
!kubectl logs metrics-server-5c474f94c4-nrwmp -n kube-system

In [18]:
!kubectl top pod

NAME                            CPU(cores)   MEMORY(bytes)   
subscription-6d5867d9cb-pn98h   3m           105Mi           


In [19]:
!kubectl top node

NAME                 CPU(cores)   CPU(%)   MEMORY(bytes)   MEMORY(%)   
kind-control-plane   126m         6%       769Mi           9%          


## Increase the load

Let's see how the autoscaler reacts to increasing the load. To do this, we can slightly modify the existing
`q6_test.py` script by putting the operator that sends the request to the subscription service into a loop.

```python
while True:
    sleep(0.1)
    response = requests.post(url, json=client).json()
    print(response)
```

Now you can run this script.

In [ ]:
!python q6_test.py

**Observation: It returns the output in a loop.**

## Question 8 (optional)

Run `kubectl get hpa subscription -hpa --watch` command to monitor how the autoscaler performs.
Within a minute or so, you should see the higher CPU load; and then - more replicas.
What was the maximum amount of the replicas during this test?


* **1**
* 2
* 3
* 4

```bash
NAME               REFERENCE                TARGETS          MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   <unknown>/20%   1         3         1          7m19s
```

>Note: It may take a few minutes to stabilize the number of replicas. Since the amount of load is not controlled
> in any way it may happen that the final number of replicas will differ from initial.


In [22]:
!kubectl get hpa subscription-hpa --watch

NAME               REFERENCE                 TARGETS   MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   3%/20%    1         3         1          65m
subscription-hpa   Deployment/subscription   23%/20%   1         3         1          70m
subscription-hpa   Deployment/subscription   42%/20%   1         3         2          70m
subscription-hpa   Deployment/subscription   146%/20%   1         3         2          71m
subscription-hpa   Deployment/subscription   143%/20%   1         3         3          71m
subscription-hpa   Deployment/subscription   125%/20%   1         3         3          71m
subscription-hpa   Deployment/subscription   97%/20%    1         3         3          71m
subscription-hpa   Deployment/subscription   44%/20%    1         3         3          72m
subscription-hpa   Deployment/subscription   3%/20%     1         3         3          72m
subscription-hpa   Deployment/subscription   2%/20%     1         3         3          72m
^C
